# Training, Testing and Choosing Best Models

The task at hand is predicting the salary of a player based on full season perfomance by taking into account their statistics. This is a typical regression task. We'll train and test:

1. Huber Regression
2. Ridge Regression
3. Random Forest

We've determined these models to be the best for the data we've gathered. 

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, RobustScaler, SQLTransformer
from pyspark.sql.functions import col
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import PipelineModel #
import time
import json

spark = SparkSession.builder \
    .appName("Model Training and Selection") \
    .getOrCreate()

In [2]:
df = spark.read.parquet("../data/processed/clean_nba.parquet", header=True, inferSchema=True)

feature_cols = [
 'age',
 'ppg',          # Points per game
 'fg_avg',       # Field goal percentage
 'three_pt_avg', # 3 Point percentege
 'ft_avg',       # Freethrow percentage
 'apg',          # Assists per game
 'rpg',          # Rebounds per game
 'spg',          # Steals per game
 'bpg',          # Blocks per game
 'mpg'           # Minutes per game (full game - 48 minutes) 
]
df[feature_cols].printSchema()

root
 |-- age: long (nullable = true)
 |-- ppg: double (nullable = true)
 |-- fg_avg: double (nullable = true)
 |-- three_pt_avg: double (nullable = true)
 |-- ft_avg: double (nullable = true)
 |-- apg: double (nullable = true)
 |-- rpg: double (nullable = true)
 |-- spg: double (nullable = true)
 |-- bpg: double (nullable = true)
 |-- mpg: double (nullable = true)



In [3]:
df = df.withColumn("age", col("age").cast("double"))
df[feature_cols].printSchema()

root
 |-- age: double (nullable = true)
 |-- ppg: double (nullable = true)
 |-- fg_avg: double (nullable = true)
 |-- three_pt_avg: double (nullable = true)
 |-- ft_avg: double (nullable = true)
 |-- apg: double (nullable = true)
 |-- rpg: double (nullable = true)
 |-- spg: double (nullable = true)
 |-- bpg: double (nullable = true)
 |-- mpg: double (nullable = true)



In [4]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='raw_features',
    handleInvalid='skip'
)
scaler = RobustScaler(
    inputCol='raw_features',
    outputCol='features',
    withScaling=True,
    withCentering=True
)
log_label = SQLTransformer(
    statement="SELECT *, log1p(salary) AS salary_log FROM __THIS__"
)
evaluator = RegressionEvaluator(
    labelCol='salary_log',
    predictionCol='prediction',
    metricName='rmse'
)

train, test = df.randomSplit([0.8, 0.2], seed=52)
print(f"Training set: {train.count()} players")
print(f"Test set: {test.count()} players")

Training set: 719 players
Test set: 208 players


## Huber Regression

In [44]:
huber = LinearRegression(
    featuresCol='features',
    labelCol='salary_log',
    loss='huber',
    standardization=True
)

huber_pipeline = Pipeline(
    stages=[
        log_label,
        assembler, 
        scaler, 
        huber
    ]
)

huber_param_grid = ParamGridBuilder() \
    .addGrid(huber.regParam, [0.01, 0.05, 0.1, 0.3, 0.5]) \
    .addGrid(huber.epsilon, [1.01, 1.35, 1.5, 2.0]) \
    .addGrid(huber.maxIter, [50, 100]) \
    .build()

huber_cv = CrossValidator(
    estimator=huber_pipeline,
    estimatorParamMaps=huber_param_grid,
    evaluator=evaluator,
    numFolds=5,
    parallelism=4,  
    seed=42
)

start_time = time.time()
huber_cv_model = huber_cv.fit(train)
huber_time = time.time() - start_time

best_huber = huber_cv_model.bestModel
print(f"\nTraining completed in {huber_time:.2f} seconds")
print(f"Best RMSE: {min(huber_cv_model.avgMetrics):.2f}")

huber_stages = best_huber.stages
huber_model = huber_stages[-1]
print("\nBest Parameters:")
print(f"  regParam: {huber_model.getRegParam()}")
print(f"  epsilon: {huber_model.getEpsilon()}")
print(f"  elasticNetParam: {huber_model.getElasticNetParam()}")
print(f"  maxIter: {huber_model.getMaxIter()}")

huber_test_pred = huber_cv_model.transform(test)
huber_test_rmse = evaluator.evaluate(huber_test_pred)
print(f"\nTest RMSE: {huber_test_rmse:.2f}")


Training completed in 126.55 seconds
Best RMSE: 0.58

Best Parameters:
  regParam: 0.05
  epsilon: 2.0
  elasticNetParam: 0.0
  maxIter: 50

Test RMSE: 0.54


In [91]:
huber_cv_model.bestModel.write().overwrite().save("models/huber_model")
print("Saved to: models/huber_model")
metadata = {
        'model_name': "huber",
        'train_time_seconds': huber_time,
        'cv_best_rmse': min(huber_cv_model.avgMetrics),
        'test_rmse': huber_test_rmse,
        'num_folds': huber_cv_model.getNumFolds(),
        'parameters': {
            "regParam": huber_model.getRegParam(),
            "epsilon": huber_model.getEpsilon(),
            "elasticNetParam": huber_model.getElasticNetParam(),
            "maxIter": huber_model.getMaxIter()
        }
    }

metadata_path = f"models/huber_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)    
print(f"Metadata saved to: {metadata_path}")

Saved to: models/huber_model
Metadata saved to: models/huber_metadata.json


## Ridge Regression

In [5]:
ridge = LinearRegression(
    featuresCol='features',
    labelCol='salary_log',
    loss='squaredError',
    elasticNetParam=0.0,  # Pure L2
    standardization=True
)
ridge_pipeline = Pipeline(stages=[
    log_label,
    assembler, 
    scaler, 
    ridge])

ridge_param_grid = ParamGridBuilder() \
    .addGrid(ridge.regParam, [0.001, 0.01, 0.1, 0.5, 1.0, 5.0]) \
    .addGrid(ridge.maxIter, [50, 100, 200]) \
    .build()

ridge_cv = CrossValidator(
    estimator=ridge_pipeline,
    estimatorParamMaps=ridge_param_grid,
    evaluator=evaluator,
    numFolds=5,
    parallelism=4,
    seed=42
)

start_time = time.time()
ridge_cv_model = ridge_cv.fit(train)
ridge_time = time.time() - start_time

best_ridge = ridge_cv_model.bestModel
print(f"\nTraining completed in {ridge_time:.2f} seconds")
print(f"Best RMSE: {min(ridge_cv_model.avgMetrics):.2f}")

ridge_stages = best_ridge.stages
ridge_model = ridge_stages[-1]
print("\nBest Parameters:")
print(f"  regParam: {ridge_model.getRegParam()}")
print(f"  maxIter: {ridge_model.getMaxIter()}")

ridge_test_pred = ridge_cv_model.transform(test)
ridge_test_rmse = evaluator.evaluate(ridge_test_pred)
print(f"\nTest RMSE: {ridge_test_rmse:.2f}")


Training completed in 55.51 seconds
Best RMSE: 0.58

Best Parameters:
  regParam: 0.01
  maxIter: 50

Test RMSE: 0.54


In [17]:
test_player={"age": 1, "ppg": 1, "fg_avg": 1, "three_pt_avg": 1, "ft_avg": 1, "apg": 1, "rpg": 1, "spg": 1, "bpg": 1, "mpg": 10}

ridge_cv_model.bestModel.transform(df.limit(1)).show()

+---------+-----------+----+-------------+------+------------------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-------------------+-----------------+---------+----------+-----------------+--------------------+--------------------+-----------------+
|     slug|  positions| age|         team|season|               ppg|             fg_avg|      three_pt_avg|            ft_avg|              apg|              rpg|               spg|                bpg|              mpg|   salary|      name|       salary_log|        raw_features|            features|       prediction|
+---------+-----------+----+-------------+------+------------------+-------------------+------------------+------------------+-----------------+-----------------+------------------+-------------------+-----------------+---------+----------+-----------------+--------------------+--------------------+-----------------+
|youngtr01|POINT GUARD|23.0|ATLANTA HAWKS| 

In [93]:
ridge_cv_model.bestModel.write().overwrite().save("models/ridge_model")
print("Saved to: models/ridge_model")
metadata = {
        'model_name': "ridge",
        'train_time_seconds': ridge_time,
        'cv_best_rmse': min(ridge_cv_model.avgMetrics),
        'test_rmse': ridge_test_rmse,
        'num_folds': ridge_cv_model.getNumFolds(),
        'parameters': {
            'regParam': ridge_model.getRegParam(),
            'maxIter': ridge_model.getMaxIter(),
        }
    }

metadata_path = f"models/ridge_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)    
print(f"Metadata saved to: {metadata_path}")

Saved to: models/ridge_model
Metadata saved to: models/ridge_metadata.json


## Random Forest

In [87]:
rf = RandomForestRegressor(
    featuresCol='features',
    labelCol='salary_log',
    seed=42
)

rf_pipeline = Pipeline(
    stages=[
        log_label,
        assembler, 
        scaler, 
        rf
    ]
)

rf_param_grid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [100, 200, 300]) \
    .addGrid(rf.maxDepth, [5, 10, 15, 20]) \
    .addGrid(rf.minInstancesPerNode, [10, 20, 30]) \
    .addGrid(rf.subsamplingRate, [0.7, 0.8, 0.9]) \
    .addGrid(rf.featureSubsetStrategy, ['sqrt', 'log2', 'onethird']) \
    .build()

rf_cv = CrossValidator(
    estimator=rf_pipeline,
    estimatorParamMaps=rf_param_grid,
    evaluator=evaluator,
    numFolds=5,
    parallelism=4,
    seed=42
)

start_time = time.time()
rf_cv_model = rf_cv.fit(train)
rf_time = time.time() - start_time

best_rf = rf_cv_model.bestModel
print(f"\nTraining completed in {rf_time:.2f} seconds")
print(f"Best RMSE: {min(rf_cv_model.avgMetrics):.2f}")

# Extract best parameters
rf_stages = best_rf.stages
rf_model = rf_stages[-1]
print("\nBest Parameters:")
print(f"  numTrees: {rf_model.getNumTrees}")
print(f"  maxDepth: {rf_model.getMaxDepth()}")
print(f"  minInstancesPerNode: {rf_model.getMinInstancesPerNode()}")
print(f"  subsamplingRate: {rf_model.getSubsamplingRate()}")
print(f"  featureSubsetStrategy: {rf_model.getFeatureSubsetStrategy()}")

# Feature importance
feature_importance = rf_model.featureImportances
print("\nFeature Importances:")
for idx, importance in enumerate(feature_importance):
    print(f"  {feature_cols[idx]}: {importance:.4f}")

# Test evaluation
rf_test_pred = rf_cv_model.transform(test)
rf_test_rmse = evaluator.evaluate(rf_test_pred)
print(f"\nTest RMSE: {rf_test_rmse:.2f}")


Training completed in 1964.85 seconds
Best RMSE: 0.56

Best Parameters:
  numTrees: 300
  maxDepth: 15
  minInstancesPerNode: 10
  subsamplingRate: 0.9
  featureSubsetStrategy: sqrt

Feature Importances:
  age: 0.1511
  ppg: 0.3007
  fg_avg: 0.0147
  three_pt_avg: 0.0176
  ft_avg: 0.0156
  apg: 0.0930
  rpg: 0.0430
  spg: 0.0585
  bpg: 0.0212
  mpg: 0.2846

Test RMSE: 0.51


In [90]:
rf_cv_model.bestModel.write().overwrite().save("models/rf_model")
print("Saved to: models/rf_model")
metadata = {
        'model_name': "rf",
        'train_time_seconds': rf_time,
        'cv_best_rmse': min(rf_cv_model.avgMetrics),
        'test_rmse': rf_test_rmse,
        'num_folds': rf_cv_model.getNumFolds(),
        'parameters': {
            "numTrees": rf_model.getNumTrees,
            "maxDepth": rf_model.getMaxDepth(),
            "minInstancesPerNode": rf_model.getMinInstancesPerNode(),
            "subsamplingRate": rf_model.getSubsamplingRate(),
            "featureSubsetStrategy": rf_model.getFeatureSubsetStrategy()
        }
    }

metadata_path = f"models/rf_metadata.json"
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)    
print(f"Metadata saved to: {metadata_path}")

Saved to: models/rf_model
Metadata saved to: models/rf_metadata.json


In [16]:
huber_cv_model = PipelineModel.load("models/huber_model")
ridge_cv_model = PipelineModel.load("models/ridge_model")
rf_cv_model = PipelineModel.load("models/rf_model")

# Model comparison

In [18]:
models = {
    'Huber': huber_cv_model,
    'Ridge': ridge_cv_model,
    'Random Forest': rf_cv_model
}

metrics = ['rmse', 'mae', 'r2', 'mse']
results = []

for model_name, model in models.items():
    predictions = model.transform(test)
    row = {'Model': model_name}
    
    for metric in metrics:
        eval = RegressionEvaluator(
            labelCol='salary_log',
            predictionCol='prediction',
            metricName=metric
        )
        score = eval.evaluate(predictions)
        row[metric.upper()] = score
    
    results.append(row)

# Convert to DataFrame and display
results_df = spark.createDataFrame(results).select(["Model",'RMSE', 'MAE', 'R2', 'MSE'])
results_df.show(truncate=False)

print("\nTraining Time Comparison:")
print(f"  Huber: {126.55} seconds") # huber_time:.2f
print(f"  Ridge: {28.32} seconds") #ridge_time
print(f"  Random Forest: {1964.85} seconds")

+-------------+------------------+-------------------+------------------+-------------------+
|Model        |RMSE              |MAE                |R2                |MSE                |
+-------------+------------------+-------------------+------------------+-------------------+
|Huber        |0.5404694674711383|0.42783862676838247|0.7209883425155825|0.2921072452685359 |
|Ridge        |0.5410502955931455|0.42994090105409016|0.7203883275049275|0.2927354223614302 |
|Random Forest|0.5051770886473097|0.3980084830275871 |0.7562372664553518|0.25520389089417184|
+-------------+------------------+-------------------+------------------+-------------------+


Training Time Comparison:
  Huber: 126.55 seconds
  Ridge: 28.32 seconds
  Random Forest: 1964.85 seconds


# Conclusions

1. Nonlinear models outperform linear ones - target relationship likely isn't simple linear.
2. Linear models still capture substantial variance - domain likely has mixed linear + nonlinear structure.
3. Random Forest = best standalone model - strong default for small tabular datasets.

# Ensamlbling these models - the idea going forward

Models differ in bias structure:

- Ridge / Huber: high-bias linear, very smooth, low variance
- Random Forest: low-bias nonlinear, higher variance, piecewise structure
- That means predictions will diverge on different parts of the feature space — this is exactly the regime where ensembles help.

Ensemble benefits:

1. Reduced bias + reduced variance because linear model corrects RF outliers, RF corrects linear underfit.
2. More stable predictions. RF alone can produce higher variance on sparse pockets of the feature space; averaging stabilizes.
3. Improves worst-case error
4. Gives us a better idea of uncertainity.
5. Linear models can extrapolate while RF learns piecewise constants. Ensamble mitigates RF failure.

We'll calculate the mean and standard deviation of the predictions from all three models for clearer results.